In [ ]:
# Inject WHL to bypass Serverless environment dependencies crash
%pip install .

In [ ]:
dbutils.widgets.text("catalog", "enterprise")
dbutils.widgets.text("schema", "bronze_market")
dbutils.widgets.text("table", "crypto_ohlc_raw")
dbutils.widgets.text("source", "coinbase")
dbutils.widgets.text("symbols", "BTCUSDT,ETHUSDT")
dbutils.widgets.text("interval", "1d")
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "today")
dbutils.widgets.text("sleep_ms", "250")
dbutils.widgets.text("max_days", "7")
dbutils.widgets.text("repair_start_date", "")
dbutils.widgets.text("repair_end_date", "")

dbutils.widgets.text("mode", "realtime")  # backfill | realtime
dbutils.widgets.text("safety_lag_minutes", "2")  
dbutils.widgets.text("state_schema", "gold_observability")
dbutils.widgets.text("state_table", "ingestion_state")
dbutils.widgets.text("default_lookback_minutes", "120")

# 解析基础参数
CATALOG    = dbutils.widgets.get("catalog").strip()
SCHEMA     = dbutils.widgets.get("schema").strip()
TABLE      = dbutils.widgets.get("table").strip()
SOURCE     = dbutils.widgets.get("source").strip()
SYMBOLS    = [s.strip().upper() for s in dbutils.widgets.get("symbols").split(",") if s.strip()]
INTERVAL   = dbutils.widgets.get("interval").strip()
START_DATE = dbutils.widgets.get("start_date").strip()
END_DATE   = dbutils.widgets.get("end_date").strip()
SLEEP_MS   = int(dbutils.widgets.get("sleep_ms"))
MAX_DAYS   = int(dbutils.widgets.get("max_days"))
REPAIR_START_DATE = dbutils.widgets.get("repair_start_date").strip()
REPAIR_END_DATE   = dbutils.widgets.get("repair_end_date").strip()

MODE = dbutils.widgets.get("mode").strip().lower()
DEFAULT_LOOKBACK_MIN = int(dbutils.widgets.get("default_lookback_minutes"))
if MODE not in ("backfill","realtime"):
    raise ValueError("mode must be backfill or realtime")

In [ ]:
import sys
from pathlib import Path
from datetime import datetime, timezone

# 1. 确保能导包 (如果挂载了 Repos)
_local_src = str(Path.cwd() / "src")
if _local_src not in sys.path:
    sys.path.insert(0, _local_src)

# 2. 导入我们重构后的核心引擎
from lakehouse.clients.coinbase import CoinbaseClient
from lakehouse.pipelines.crypto_bronze_ingestor import CryptoBronzeIngestor

# 辅助方法：快速解析 Widget 输入的日子
def parse_yyyy_mm_dd(s: str) -> datetime:
    if not s or str(s).strip().lower() == "today":
        return datetime.now(timezone.utc)
    return datetime.strptime(str(s).strip(), "%Y-%m-%d").replace(tzinfo=timezone.utc)

start_dt = parse_yyyy_mm_dd(START_DATE) if START_DATE else None
end_dt = parse_yyyy_mm_dd(END_DATE)
repair_start = parse_yyyy_mm_dd(REPAIR_START_DATE) if REPAIR_START_DATE else None
repair_end = parse_yyyy_mm_dd(REPAIR_END_DATE) if REPAIR_END_DATE else None

# 3. 组装并启动 Pipeline
client = CoinbaseClient(sleep_ms=SLEEP_MS)

ingestor = CryptoBronzeIngestor(
    spark=spark,                # Notebook 自带的 Spark Session
    catalog=CATALOG,
    schema=SCHEMA,
    table_name=TABLE,
    api_client=client,          # 依赖注入：把 Coinbase 的逻辑塞给 Orchestrator
    source_name=SOURCE
)

# 一键执行！
result = ingestor.run(
    mode=MODE,
    symbols=SYMBOLS,
    interval=INTERVAL,
    end_date=end_dt,
    start_date=start_dt,
    max_days=MAX_DAYS,
    default_lookback_days=DEFAULT_LOOKBACK_MIN // 1440 or 1,
    repair_start_date=repair_start,
    repair_end_date=repair_end
)

print(f"[SUCCESS] Pipeline completed: {result}")